# Orbit Wars — Gymnasium Wrapper

A single-agent [Gymnasium](https://gymnasium.farama.org/api/env/) interface to the Kaggle `orbit_wars` environment.

## Game mechanics
- **100 × 100** board with a sun at the centre (radius 10) that destroys any fleet that crosses it
- **Planets** produce ships every turn; inner planets rotate around the sun, outer planets are static
- Your agent returns a list of **moves**: `[from_planet_id, angle_radians, num_ships]`
- **Fleets** fly in a straight line at the given angle; speed scales with fleet size (1 ship = 1 unit/turn, up to 6 units/turn)
- **Combat**: arriving ships subtract from the defender's garrison; ownership flips when the garrison goes negative
- **Comets**: temporary planets that drift through on elliptical paths and can be captured
- **Win condition**: most total ships (planets + fleets) when the 500-step limit is reached, or last player standing

## Observation space (Dict)
| Key | Shape | Dtype | Description |
|---|---|---|---|
| `planets` | `(MAX_PLANETS, 7)` | float32 | `[id, owner, x, y, radius, ships, production]`; zero-padded; owner `-1` = neutral |
| `n_planets` | `(1,)` | int32 | number of active planets |
| `fleets` | `(MAX_FLEETS, 7)` | float32 | `[id, owner, x, y, angle, from_planet_id, ships]`; zero-padded |
| `n_fleets` | `(1,)` | int32 | number of active fleets |
| `player` | `(1,)` | int32 | agent's player index (always 0) |
| `angular_velocity` | `(1,)` | float32 | inner-planet rotation speed (rad/turn) |
| `step` | `(1,)` | int32 | current episode step (0-based) |

## Action space: `Box(0, 1, shape=(MAX_PLANETS, 3))`
Row `i` maps to the planet in `obs["planets"][i]`.
| Column | Name | Description |
|---|---|---|
| 0 | `send` | > 0.5 launches a fleet from this planet this turn |
| 1 | `angle` | launch direction, normalised `[0, 1]` → multiplied by `2π` internally |
| 2 | `fraction` | fraction of available ships to send `[0, 1]` |

Rows for unowned planets or empty padding slots are ignored.

## Rewards
- `+1.0` on win, `-1.0` on loss/draw, `0.0` every non-terminal step (sparse)
- Pass `reward_shaping=True` to add a small per-step signal: `0.01 × (agent_ships − max_opponent_ships) / 1000`

## Install dependencies

In [1]:
import sys
!{sys.executable} -m pip install --upgrade "kaggle-environments>=1.28.0" gymnasium numpy

## Implementation

In [2]:
import math
from typing import Any

import numpy as np
import gymnasium as gym
from gymnasium import spaces
from kaggle_environments import make
from kaggle_environments.envs.orbit_wars.orbit_wars import agents as _orbit_wars_agents

MAX_PLANETS = 64   # 40 regular + up to 20 comets (5 spawns × 4) + buffer
MAX_FLEETS = 256

Loading environment cabt failed: dlopen(/Users/gauravmehra/development/kaggle/orbitwars/.venv/lib/python3.14/site-packages/kaggle_environments/envs/cabt/cg/libcg.so, 0x0006): tried: '/Users/gauravmehra/development/kaggle/orbitwars/.venv/lib/python3.14/site-packages/kaggle_environments/envs/cabt/cg/libcg.so' (slice is not valid mach-o file), '/System/Volumes/Preboot/Cryptexes/OS/Users/gauravmehra/development/kaggle/orbitwars/.venv/lib/python3.14/site-packages/kaggle_environments/envs/cabt/cg/libcg.so' (no such file), '/Users/gauravmehra/development/kaggle/orbitwars/.venv/lib/python3.14/site-packages/kaggle_environments/envs/cabt/cg/libcg.so' (slice is not valid mach-o file)
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 19.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spi

In [3]:
class OrbitWarsEnv(gym.Env):
    """Single-agent Gymnasium interface to orbit_wars.

    The agent always plays as player 0. All other players are driven by
    opponent policies that are called automatically inside step().

    Args:
        opponent: A built-in agent name ("random" / "starter"), a callable
                  with signature fn(obs_dict) -> [[planet_id, angle, ships], ...],
                  or a list of either for multi-player games. Lists are
                  padded to num_players-1 using the last element.
        num_players: 2 or 4.
        render_mode: None, "ansi" (return string), or "human" (print to stdout).
        reward_shaping: If True, add a small intermediate reward each step
                        proportional to the agent's ship advantage.
    """

    metadata = {"render_modes": ["ansi", "human"], "render_fps": 1}

    def __init__(
        self,
        opponent: Any = "random",
        num_players: int = 2,
        render_mode: str | None = None,
        reward_shaping: bool = False,
    ):
        super().__init__()
        assert num_players in (2, 4), "num_players must be 2 or 4"
        assert render_mode is None or render_mode in self.metadata["render_modes"]

        self.num_players = num_players
        self.render_mode = render_mode
        self.reward_shaping = reward_shaping
        self.spec = None

        raw = opponent if isinstance(opponent, (list, tuple)) else [opponent]
        self._opponents: list = [self._resolve_agent(o) for o in raw]
        while len(self._opponents) < num_players - 1:
            self._opponents.append(self._opponents[-1])

        # Planets: [id, owner, x, y, radius, ships, production]
        # Fleets:  [id, owner, x, y, angle, from_planet_id, ships]
        self.observation_space = spaces.Dict({
            "planets": spaces.Box(-1.0, 10_000.0, shape=(MAX_PLANETS, 7), dtype=np.float32),
            "n_planets": spaces.Box(0, MAX_PLANETS, shape=(1,), dtype=np.int32),
            "fleets": spaces.Box(-1.0, 10_000.0, shape=(MAX_FLEETS, 7), dtype=np.float32),
            "n_fleets": spaces.Box(0, MAX_FLEETS, shape=(1,), dtype=np.int32),
            "player": spaces.Box(0, 3, shape=(1,), dtype=np.int32),
            "angular_velocity": spaces.Box(0.0, 0.1, shape=(1,), dtype=np.float32),
            "step": spaces.Box(0, 500, shape=(1,), dtype=np.int32),
        })

        # Per planet slot: [send, angle_norm, ships_fraction]
        self.action_space = spaces.Box(0.0, 1.0, shape=(MAX_PLANETS, 3), dtype=np.float32)

        self._env = None
        self._planet_slots: dict[int, int] = {}  # planet_id → obs row index

    # ------------------------------------------------------------------
    # Gymnasium core API
    # ------------------------------------------------------------------

    def reset(self, *, seed: int | None = None, options: dict | None = None):
        super().reset(seed=seed)
        config = {"seed": seed} if seed is not None else {}
        self._env = make("orbit_wars", configuration=config, debug=False)
        self._env.reset()
        obs = self._build_obs()
        return obs, self._build_info()

    def step(self, action: np.ndarray):
        assert self._env is not None, "Call reset() before step()"

        # Player 0's action decoded from the Box array
        all_actions: list = [None] * self.num_players
        all_actions[0] = self._decode_action(action, player_id=0)

        # Opponents act on their own observations
        for opp_idx, opp_fn in enumerate(self._opponents):
            pid = opp_idx + 1
            opp_obs = self._env.state[pid].observation
            all_actions[pid] = opp_fn(self._obs_to_dict(opp_obs))

        self._env.step(all_actions)

        agent_state = self._env.state[0]
        terminated = agent_state.status == "DONE"
        reward = float(agent_state.reward) if terminated else 0.0

        if self.reward_shaping:
            reward += self._shaping_reward()

        return self._build_obs(), reward, terminated, False, self._build_info()

    def render(self):
        if self._env is None:
            return None
        text = self._env.render(mode="ansi")
        if self.render_mode == "human":
            print(text)
            return None
        return text

    def close(self):
        self._env = None

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _resolve_agent(agent):
        """Accept a named built-in string or any callable."""
        if callable(agent):
            return agent
        if agent in _orbit_wars_agents:
            return _orbit_wars_agents[agent]
        raise ValueError(
            f"Unknown agent '{agent}'. Available: {list(_orbit_wars_agents.keys())}"
        )

    def _build_obs(self) -> dict:
        raw = self._env.state[0].observation
        planets_raw = list(getattr(raw, "planets", []) or [])
        fleets_raw = list(getattr(raw, "fleets", []) or [])

        planet_arr = np.zeros((MAX_PLANETS, 7), dtype=np.float32)
        self._planet_slots = {}
        n_planets = min(len(planets_raw), MAX_PLANETS)
        for i, p in enumerate(planets_raw[:MAX_PLANETS]):
            planet_arr[i] = p
            self._planet_slots[int(p[0])] = i

        fleet_arr = np.zeros((MAX_FLEETS, 7), dtype=np.float32)
        n_fleets = min(len(fleets_raw), MAX_FLEETS)
        for i, f in enumerate(fleets_raw[:MAX_FLEETS]):
            fleet_arr[i] = f

        return {
            "planets": planet_arr,
            "n_planets": np.array([n_planets], dtype=np.int32),
            "fleets": fleet_arr,
            "n_fleets": np.array([n_fleets], dtype=np.int32),
            "player": np.array([int(getattr(raw, "player", 0))], dtype=np.int32),
            "angular_velocity": np.array(
                [float(getattr(raw, "angular_velocity", 0.0))], dtype=np.float32
            ),
            "step": np.array([int(getattr(raw, "step", 0))], dtype=np.int32),
        }

    def _build_info(self) -> dict:
        s = self._env.state[0]
        return {"status": s.status, "reward": float(s.reward)}

    def _decode_action(self, action: np.ndarray, player_id: int) -> list:
        """Convert the Box action array to a kaggle move list.

        Kaggle format: [[from_planet_id, angle_radians, num_ships], ...]
        The interpreter validates: planet must be owned by this player,
        ships must be > 0 and <= current garrison.
        """
        raw = self._env.state[player_id].observation
        planets_raw = list(getattr(raw, "planets", []) or [])
        moves = []
        for p in planets_raw:
            pid, owner, _, _, _, ships, _ = p
            pid, owner, ships = int(pid), int(owner), int(ships)
            if owner != player_id or ships <= 0:
                continue
            slot = self._planet_slots.get(pid)
            if slot is None or slot >= MAX_PLANETS:
                continue
            send, angle_norm, ships_frac = action[slot]
            if float(send) <= 0.5:
                continue
            angle = float(angle_norm) * 2.0 * math.pi   # [0,1] → [0, 2π]
            n_ships = max(1, int(float(ships_frac) * ships))
            moves.append([pid, angle, n_ships])
        return moves

    def _shaping_reward(self) -> float:
        """Small intermediate reward: agent ship advantage over best opponent."""
        raw = self._env.state[0].observation
        planets_raw = list(getattr(raw, "planets", []) or [])
        fleets_raw = list(getattr(raw, "fleets", []) or [])

        ship_counts: dict[int, int] = {}
        for p in planets_raw:
            owner = int(p[1])
            if owner >= 0:
                ship_counts[owner] = ship_counts.get(owner, 0) + int(p[5])
        for f in fleets_raw:
            owner = int(f[1])
            ship_counts[owner] = ship_counts.get(owner, 0) + int(f[6])

        my_ships = ship_counts.get(0, 0)
        opp_ships = max((v for k, v in ship_counts.items() if k != 0), default=0)
        return 0.01 * (my_ships - opp_ships) / 1000.0

    @staticmethod
    def _obs_to_dict(obs) -> dict:
        """Convert a kaggle SimpleNamespace observation to a plain dict.

        Built-in agents and getting-started tutorial agents use obs.get()
        style access, so they all expect a dict.
        """
        return {
            "player": getattr(obs, "player", 0),
            "planets": list(getattr(obs, "planets", []) or []),
            "fleets": list(getattr(obs, "fleets", []) or []),
            "angular_velocity": getattr(obs, "angular_velocity", 0.0),
            "initial_planets": list(getattr(obs, "initial_planets", []) or []),
            "next_fleet_id": getattr(obs, "next_fleet_id", 0),
            "comets": list(getattr(obs, "comets", []) or []),
            "comet_planet_ids": list(getattr(obs, "comet_planet_ids", []) or []),
            "step": getattr(obs, "step", 0),
            "remainingOverageTime": getattr(obs, "remainingOverageTime", 60),
        }

## Smoke test

Create the env, run a few steps with random actions, and confirm the API works end-to-end.

In [4]:
env = OrbitWarsEnv(opponent="random", render_mode="ansi")

obs, info = env.reset(seed=42)
print("Observation keys :", list(obs.keys()))
print(f"Active planets   : {obs['n_planets'][0]}")
print(f"Active fleets    : {obs['n_fleets'][0]}")
print(f"Angular velocity : {obs['angular_velocity'][0]:.4f} rad/turn")

total_reward = 0.0
for t in range(10):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    print(f"  step {t+1:3d} | planets={obs['n_planets'][0]:2d}  "
          f"fleets={obs['n_fleets'][0]:3d}  reward={reward:+.3f}  done={terminated}")
    if terminated:
        break

print(f"\nTotal reward: {total_reward:+.1f}")
env.close()

Observation keys : ['planets', 'n_planets', 'fleets', 'n_fleets', 'player', 'angular_velocity', 'step']
Active planets   : 20
Active fleets    : 0
Angular velocity : 0.0410 rad/turn
  step   1 | planets=20  fleets=  1  reward=+0.000  done=False
  step   2 | planets=20  fleets=  2  reward=+0.000  done=False
  step   3 | planets=20  fleets=  3  reward=+0.000  done=False
  step   4 | planets=20  fleets=  3  reward=+0.000  done=False
  step   5 | planets=20  fleets=  3  reward=+0.000  done=False
  step   6 | planets=20  fleets=  4  reward=+0.000  done=False
  step   7 | planets=20  fleets=  3  reward=+0.000  done=False
  step   8 | planets=20  fleets=  4  reward=+0.000  done=False
  step   9 | planets=20  fleets=  5  reward=+0.000  done=False
  step  10 | planets=20  fleets=  6  reward=+0.000  done=False

Total reward: +0.0


## Using a custom opponent

You can pass any callable as the opponent — here we reuse the `nearest_planet_sniper` from the getting-started notebook.

In [5]:
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

def nearest_planet_sniper(obs):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]

    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]
    if not targets:
        return moves

    for mine in my_planets:
        nearest = min(targets, key=lambda t: math.hypot(mine.x - t.x, mine.y - t.y))
        ships_needed = max(nearest.ships + 1, 20)
        if mine.ships >= ships_needed:
            angle = math.atan2(nearest.y - mine.y, nearest.x - mine.x)
            moves.append([mine.id, angle, ships_needed])
    return moves


env = OrbitWarsEnv(opponent=nearest_planet_sniper)
obs, info = env.reset(seed=0)

terminated = False
while not terminated:
    action = env.action_space.sample()
    obs, reward, terminated, _, info = env.step(action)

print(f"Episode finished | status={info['status']}  final_reward={reward:+.1f}")
env.close()

Episode finished | status=DONE  final_reward=+1.0


In [8]:
from stable_baselines3.common.env_checker import check_env                        
   
check_env(OrbitWarsEnv())   

/Users/gauravmehra/development/kaggle/orbitwars/.venv/lib/python3.14/site-packages/stable_baselines3/common/env_checker.py:324: UserWarning: Your observation fleets has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(
/Users/gauravmehra/development/kaggle/orbitwars/.venv/lib/python3.14/site-packages/stable_baselines3/common/env_checker.py:324: UserWarning: Your observation planets has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(
/Users/gauravmehra/development/kaggle/orbitwars/.venv/lib/python3.14/site-packages/stable_baselines3/common/env_checker.py:515: UserWarning: We recommend you to use a symmetric and normalized Box action space (range=[-1, 1]) cf. https://stable-baselines3.readth